In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "common.py").exists():
    AI_DIR = cwd
elif (cwd.parent / "common.py").exists():
    AI_DIR = cwd.parent
elif (cwd / "AI" / "common.py").exists():
    AI_DIR = cwd / "AI"
else:
    raise FileNotFoundError("Cannot locate AI/common.py")

if str(AI_DIR) not in sys.path:
    sys.path.insert(0, str(AI_DIR))

from common import FEATURE_COLUMNS, fetch_dataset_frame, load_device_id, load_model, validate_feature_columns


In [2]:
model = load_model()
print("Model loaded")


Model loaded


In [3]:
device_id = load_device_id()
df = fetch_dataset_frame(device_id=device_id)
if df.empty:
    df = fetch_dataset_frame()

if df.empty:
    raise ValueError("Dataset is empty. Cannot test model.")

print("Testing device:", device_id)
print("Total rows:", len(df))
df.tail()


Total rows: 1230


,delta_gas,gas,gas_relative,hour,humidity,label,month,temperature,timestamp
1225,5,737,1.01368,11,53.22762,1,3,32.69863,1773892858
1226,-2,735,1.00982,11,53.30582,1,3,32.70893,1773892859
1227,1,736,1.01007,11,53.24907,1,3,32.70073,1773892860
1228,-1,735,1.00782,11,53.23315,1,3,32.69100,1773892861
1229,-1,734,1.00580,11,53.28341,1,3,32.71198,1773892862


In [4]:
validate_feature_columns(df)
if "label" not in df.columns:
    raise ValueError("Missing label column in dataset.")

if "timestamp" in df.columns:
    latest = df.sort_values("timestamp").tail(1)
else:
    latest = df.tail(1)

print("Latest sensor data:")
if "device_id" in latest.columns:
    print("Latest device:", latest["device_id"].iloc[0])
latest


Latest sensor data:


,delta_gas,gas,gas_relative,hour,humidity,label,month,temperature,timestamp
1229,-1,734,1.0058,11,53.28341,1,3,32.71198,1773892862


In [5]:
X = latest[FEATURE_COLUMNS]
real_label = latest["label"].iloc[0]
pred = model.predict(X)[0]

if "device_id" in latest.columns:
    print("Device ID:", latest["device_id"].iloc[0])

print("ESP32 label:", real_label)
print("AI prediction:", pred)
print("AI MATCHED SENSOR LOGIC" if pred == real_label else "AI DIFFERENT FROM SENSOR")


ESP32 label: 1
AI prediction: 1
AI MATCHED SENSOR LOGIC
